# SmolVLA 実GPU推論（2軸・日本語入力）

- **目的：** Hugging Faceの実際の `lerobot/smolvla_base` の重みをGPUへロードし、カメラに相当する人工画像と日本語指示から **2軸×行動チャンク** を生成する。
- **注意：** この検証は学習前のベースモデルの形状確認です。日本語理解や目標への正しい照準、実機成功率は検証しません。
- Colabのメニュー **ランタイム → ランタイムのタイプを変更 → GPU** を設定し、上から順にセルを実行してください。
- すべての出力が実行記録としてノートブック内に残ります。

In [ ]:
import sys, subprocess
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "ColabでGPUランタイムを選択してください"
print("GPU:", torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable,"-m","pip","install","-q","lerobot[smolvla]==0.6.1"])
print("PASS: GPU and lerobot installation")

## 学習済み重みを読み込む

`SmolVLAConfig` を**モデル構築前**に2軸入力・出力へ変更します。モデルの内部パディング次元は32のままです。

合成画像と合成統計量は、実機の学習データではありません。出力された角度は信頼できないため、Picoに送らないでください。

In [ ]:
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.pretrained import FeatureType, PolicyFeature
from lerobot.policies.factory import make_pre_post_processors

MODEL = "lerobot/smolvla_base"
cfg=SmolVLAConfig.from_pretrained(MODEL)
cfg.device="cuda"
cfg.input_features={
    "observation.state":PolicyFeature(type=FeatureType.STATE,shape=(2,)),
    "observation.images.front":PolicyFeature(type=FeatureType.VISUAL,shape=(3,96,96))
}
cfg.output_features={"action":PolicyFeature(type=FeatureType.ACTION,shape=(2,))}
assert cfg.max_state_dim >= 2 and cfg.max_action_dim >= 2
policy=SmolVLAPolicy.from_pretrained(MODEL,config=cfg).eval()
assert policy.config.action_feature.shape[0] == 2
assert policy.config.state_feature.shape[0] == 2
print("PASS: real pretrained weights loaded; config input/output dimensionality = 2")

## 同じ画像に対して赤・青の日本語指示を送信

推論の乱数シードを同じにして、指示だけを切り替えます。単なる**2軸出力の成功**と、言語を理解して正しい方向を選ぶことは区別します。

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
img=Image.new("RGB",(96,96),(235,236,230))
draw=ImageDraw.Draw(img)
draw.ellipse((18,40,34,56),fill=(234,56,50))
draw.ellipse((62,24,78,40),fill=(47,103,235))
print("synthetic scene:")
display(img)

stats={
    "observation.state":{"mean":torch.tensor([90.,90.]),"std":torch.tensor([30.,30.])},
    "action":{"mean":torch.tensor([90.,90.]),"std":torch.tensor([30.,30.])},
}
preprocess,postprocess=make_pre_post_processors(cfg,dataset_stats=stats)
observation={
    "observation.state":torch.tensor([90.,90.],dtype=torch.float32),
    "observation.images.front":torch.from_numpy(np.asarray(img).copy()).permute(2,0,1).float()/255.,
}
results={}
for instruction in ["赤い丸を指して","青い丸を指して"]:
    torch.manual_seed(123)
    entry={**observation,"task":instruction}
    batch=preprocess(entry)
    with torch.inference_mode():
        action_chunk=policy.predict_action_chunk(batch)
    assert action_chunk.ndim==3,action_chunk.shape
    assert action_chunk.shape[0]==1 and action_chunk.shape[2]==2,action_chunk.shape
    assert torch.isfinite(action_chunk).all(),"NaN/Inf actions"
    results[instruction]=action_chunk.detach().float().cpu()
    print(instruction,"shape:",tuple(action_chunk.shape))
    print("first three NORMALIZED actions:",results[instruction][0,:3].numpy())
print("PASS: SmolVLA actually generated a 2-axis action chunk on GPU")
print("NOTE: actions are untrained for this 2-DOF setup; DO NOT send these to motors")

## 機械判読ログ

学習済みSmolVLAの呼出しに成功したか、GPU名・形状をJSONへ保存します。推論値自体は教師なしのため正しい角度ではありません。

In [ ]:
import json,datetime
report={"timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "gpu":torch.cuda.get_device_name(0),"model":MODEL,
        "actual_pretrained_weights":True,"finetuned_for_pico":False,
        "instructions":list(results.keys()),
        "shapes":{k:list(v.shape) for k,v in results.items()},
        "note":"This is dimensionality/inference smoke test only; not target-pointing accuracy."}
path="/content/smolvla_gpu_smoke_result.json"
with open(path,"w",encoding="utf-8") as f:json.dump(report,f,ensure_ascii=False,indent=2)
print(json.dumps(report,ensure_ascii=False,indent=2))
print("saved:",path)